# Xuan Thuy public reproduction launcher

This notebook contains no model, loss, split, or metric implementation. It checks out an immutable public release and invokes the versioned `xtseg` pipeline. Data and run artifacts remain in your own Google Drive.

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/hongphuc-maker/xuan-thuy-dmi-segmentation.git'
GIT_REF = 'v0.1.0'
DESTINATION = Path('/content/xuan-thuy-dmi-segmentation')

if DESTINATION.exists():
    raise FileExistsError(f'{DESTINATION} already exists; use a fresh runtime or inspect it explicitly')
subprocess.run([
    'git', 'clone', '--branch', GIT_REF, '--depth', '1',
    REPOSITORY_URL, str(DESTINATION),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(DESTINATION / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(DESTINATION)], check=True)

SOURCE_ROOT = DESTINATION / 'src'
sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
import xuanthuy_seg
if GIT_REF.startswith('v') and xuanthuy_seg.__version__ != GIT_REF[1:]:
    raise RuntimeError(f'Package version {xuanthuy_seg.__version__} does not match {GIT_REF}')
print('Checked out', GIT_REF, '| package', xuanthuy_seg.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXPERIMENT_NAME = 'pipeline_e0w_vertical.yaml'
EXPERIMENT = DESTINATION / 'configs' / 'experiments' / EXPERIMENT_NAME
DATA_ROOT = Path('/content/drive/MyDrive/xuan_thuy_dmi/data')
RUN_ROOT = Path('/content/drive/MyDrive/xuan_thuy_dmi/runs/weighted_ce')
INITIALIZATION_CHECKPOINT = None  # Required for a CE-to-DMI YAML.
DEVICE = 'cuda'

print('Experiment:', EXPERIMENT)
print('Data root:', DATA_ROOT)
print('Run root:', RUN_ROOT)

In [ ]:
from xuanthuy_seg.cli import main as xtseg_main

def pipeline_arguments(through):
    arguments = [
        'run-pipeline',
        '--experiment', str(EXPERIMENT),
        '--data-root', str(DATA_ROOT),
        '--run-root', str(RUN_ROOT),
        '--through', through,
        '--device', DEVICE,
    ]
    if INITIALIZATION_CHECKPOINT is not None:
        arguments.extend(['--initialization-checkpoint', str(INITIALIZATION_CHECKPOINT)])
    return arguments

if xtseg_main(['validate-config', '--experiment', str(EXPERIMENT)]) != 0:
    raise RuntimeError('Configuration validation failed')
if xtseg_main([
    'verify-data', '--experiment', str(EXPERIMENT),
    '--data-root', str(DATA_ROOT), '--roles', 'image', 'label',
]) != 0:
    raise RuntimeError('Image/label integrity verification failed')

In [ ]:
# Prepare artifacts, train, or auto-resume after a runtime interruption.
status = xtseg_main(pipeline_arguments('train'))
if status != 0:
    raise RuntimeError(f'Training pipeline failed with exit code {status}')

## Run only after checkpoint selection is frozen

The next cell creates the georeferenced full-scene map from `FINAL_MODEL.pt`.

In [ ]:
status = xtseg_main(pipeline_arguments('map'))
if status != 0:
    raise RuntimeError(f'Map pipeline failed with exit code {status}')

## Independent verified-point evaluation

Keep this locked during development. Set the confirmation only after the protocol, checkpoint, and map have been selected without looking at the independent points.

In [ ]:
CONFIRM_INDEPENDENT_EVALUATION = False
if not CONFIRM_INDEPENDENT_EVALUATION:
    raise RuntimeError('Independent evaluation remains sealed')
status = xtseg_main(
    pipeline_arguments('evaluate') + ['--confirm-independent-evaluation']
)
if status != 0:
    raise RuntimeError(f'Independent evaluation failed with exit code {status}')